# LLM Learned Operations

Small LLMs are notoriously bad with maths (which is funny because one forward pass through a transfomer model has many many matrix multiplications and additions!). We can use our `SymbolicModel` to probe what functions a small LLM is actually using when carrying out mathematical operations.

In this demo, we use the small model Llama-3.2-1B-Instruct. Depending on your laptop, you should be able to run this whole notebook locally!

## Set-up

In [1]:
import numpy as np
from symtorch import SymbolicModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# This is the model we are going to use
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load the tokenizer and the model
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.generation_config.pad_token_id = tok.eos_token_id

torch.manual_seed(290402)
# For our experiment, we want a deterministic model
torch.use_deterministic_algorithms(True)

# Function which calls our LLM
def llm_call(prompt: str, max_tokens = 250) -> str:
    inputs = tok(prompt, return_tensors="pt")

    out = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,          # greedy
    )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

Let's try out our LLM to see how it performs at basic addition.

In [119]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 12+7=?")

In [120]:
print(output)

.

## Step 1: We need to add 12 and 7 together.
## Step 2: The result of the addition is 19.
## Step 3: We need to put the result in the format $boxed$.
## Step 4: The final answer is $\boxed{19}$.

The final answer is: $\boxed{19}$


For smaller numbers it can perform reasonably well. Let's see it's behvaiour for larger (3 digit) numbers.

In [84]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 972+373=?")
print(output)

print("True answer = ", 972+373)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


## Step 1: Add the two numbers together
First, we need to add 972 and 373 together.

## Step 2: Calculate the sum
972 + 373 = 1445

## Step 3: Format the answer
The answer should be in the format $boxed{1445}$.

The final answer is: $\boxed{1445}$
True answer =  1345


No longer performs that great! 

We can use `SymbolicModel` to approximate the functions that the LLM is using when performing maths.

In [23]:
# Get out the number outputted by llm as float
def extract_boxed_number(text: str) -> float:
    def parse_number(s: str) -> float:
        return float(s.replace(',', ''))
    
    # Try $\boxed{...}$ format first
    match = re.search(r'\$\\boxed\{([^}]+)\}\$', text)
    if match:
        return parse_number(match.group(1))
    # Try $boxed{...}$ format (without backslash)
    match = re.search(r'\$boxed\{([^}]+)\}\$', text)
    if match:
        return parse_number(match.group(1))
    # Try \boxed{...} without dollar signs
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return parse_number(match.group(1))
    # Try boxed{...} without anything
    match = re.search(r'boxed\{([^}]+)\}', text)
    if match:
        return parse_number(match.group(1))
    # Try $number$ format (without boxed)
    match = re.search(r'\$([0-9,.]+)\$', text)
    if match:
        return parse_number(match.group(1))
    # Fallback: try to find any number after an equals sign
    match = re.search(r'=\s*([\d,]+)', text)
    if match:
        return parse_number(match.group(1))
    # Fallback: try to find "Answer: number"
    match = re.search(r'Answer:\s*([\d,.]+)', text)
    if match:
        return parse_number(match.group(1))
    raise ValueError(f"No boxed number found in: {text}")

# Function to create a dataset of random number pairs 
def random_number_pairs(N = 100, maximum = 999):
    return np.random.randint(0, maximum, size=(N, 2))

## Addition

`SymbolicModel` is model-agnostic. You just need to pass a function that is of the form `f(inputs) = outputs`.

In [96]:
# Create a function that the SymbolicModel expects 
def llm_addition(X):
    outputs = []
    # X is of shape (N,2)
    for n in range(X.shape[0]):
        a = X[n,0]
        b = X[n,1]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(a)}+{int(b)}=?")
        output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

Create a random dataset of numbers to add.

In [97]:
np.random.seed(290402)

X = random_number_pairs(50)

Example of the numbers in our dataset.

In [98]:
print(X[:5,:])

[[451  41]
 [871 582]
 [237 193]
 [661 992]
 [417 724]]


In [ ]:
# Initialise our model
symbolic_model_addition = SymbolicModel(sr_params={'constraints': {'sin':1, 'exp':1}, 'niterations' : 1000})

In [ ]:
#Perform SR on our model
symbolic_model_addition.fit(llm_addition, X)

In [101]:
symbolic_model_addition.get_equation()

'(x0 + ((inv(x0 + -938.28375) * x1) + x1)) + (inv(sin(x0) + 0.1123411) * 1.4623601)'

`symbolic_model_addition` contains a list of equations. The more complex equations fit the inputs $\rightarrow$ outputs better, but may overfit. The 'best equation' is the one that balances complexity and accuracy the most (largest gain in accuracy per increase in complexity).

Let's see how the LLM performs other tasks.

## Multiplication

In [ ]:
def llm_multiplication(X):
    outputs = []
    # X is of shape (N,2)
    for n in range(X.shape[0]):
        a = X[n,0]
        b = X[n,1]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(a)} * {int(b)}=?")
        output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [ ]:
# Initialise our model
symbolic_model_multiplication = SymbolicModel(sr_params={'constraints': {'sin':1, 'exp':1}, 'niterations' : 1000})

In [ ]:
#Perform SR on our model
symbolic_model_multiplication.fit(llm_multiplication, X)

In [116]:
symbolic_model_multiplication.get_equation()

'x0 * ((inv(-0.94997066 + ((x0 + -0.016344598) * inv(x1))) * -0.11236402) + x1)'

## Counting

What does the LLM return when counting the number of 1s in a string of 1s and 0s?

In [195]:
extract_boxed_number(llm_call("Return only the numeric answer in the format $boxed$. How many 1s are there in the string 000101", max_tokens= 250))

4.0

In [4]:
def random_number_string_01(N = 100, len_sequence = 4):
    return np.random.randint(0, 2, size=(N, len_sequence))

X_counts_01 = random_number_string_01(N = 25)

In [5]:
def llm_counting(X):
    outputs = []
    # X is of shape (N,10)
    for n in range(X.shape[0]):
        sequence = ''.join(map(str, X[n,:]))
        # print(sequence)
        output = llm_call(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}", max_tokens=250)
        # print(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}")

        try:
            output = extract_boxed_number(output)
        except ValueError:
            print("No boxed number found. Trying again with more tokens.")
            output = llm_call(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}", max_tokens=500)
            output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [6]:
# Initialise our model
symbolic_model_counting = SymbolicModel(sr_params={'constraints': {'sin':1, 'exp':1}, 'niterations' : 1000})

In [7]:
symbolic_model_counting.fit(llm_counting, X_counts_01)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


Running SR on output dimension 0 of 0


[ Info: Started!



Expressions evaluated per second: 1.960e+06
Progress: 11291 / 31000 total iterations (36.423%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.840e+00  0.000e+00  y = 2.4
5           1.484e+00  5.369e-02  y = (x₃ * -1.2013) + 2.9286
7           1.426e+00  1.999e-02  y = ((x₁ + -1.474) * x₃) + 2.9286
9           1.350e+00  2.732e-02  y = (x₃ * -1.0066) + ((x₁ * x₀) + 2.6429)
11          1.138e+00  8.536e-02  y = (((x₁ + x₀) + -1.8974) * (x₃ + x₁)) + 3.1539
12          5.530e-01  7.221e-01  y = (inv(x₃ + (x₁ + -0.36405)) * (x₀ + -0.71498)) + 2.5049
14          5.160e-01  3.467e-02  y = (((x₁ + -0.75314) * inv((x₀ + x₃) + -0.30124)) + x₀) +...
                                       2.0312
16          4.302e-01  9.090e-02  y = (inv((x₀ + (x₃ * (0.75411 + x₂))) + -0.32153) * (x₁ + ..

[ Info: Final population:
[ Info: Results saved to:


  - SR_output/SymbolicModel/dim0_1763996946/hall_of_fame.csv


The LLM is really terrible at counting! The equations it learns are not remotely what you would expect ($x_0+x_1+...+x_N$).

In [8]:
symbolic_model_counting.get_equation()

'((((x1 * 1.3154923) + -0.81516516) * inv((x0 + ((x3 + (x1 + -0.085671134)) * ((x2 * -1.6954852) + 1.3834041))) + -0.28615704)) + x0) + 1.8765043'

## Temperature conversion

Let's see how the LLM calculates Celsius to Fahrenheit. We would expect $y = \frac{9}{5}x + 32$.

In [11]:
llm_call("Return only the numeric answer in the format $boxed$. What is 30 degrees Celsius in Fahrenheit?")

"To convert Celsius to Fahrenheit, multiply the Celsius temperature by 9/5 and add 32. Here's the formula: $F = \\frac{9}{5}C + 32$ where $C$ is the temperature in Celsius. Plug in the value of $C$ and solve for $F$. $F = \\frac{9}{5}(30) + 32$ $F = \\frac{270}{5} + 32$ $F = 54 + 32$ $F = 86$ Therefore, 30 degrees Celsius is equal to 86 degrees Fahrenheit."

First, let's try with temperatures that are within a regular range (ie. between -20 and 200C).

In [19]:
def llm_C_to_F(X):
    outputs = []
    # X is of shape (N,1)
    for n in range(X.shape[0]):
        temp_C = X[n,0]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(temp_C)} degreees Celsius in Fahrenheit?")
        try:
            output = extract_boxed_number(output)
        except ValueError:
            print("No boxed number found. Trying again with more tokens.")
            output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(temp_C)} degreees Celsius in Fahrenheit?", max_tokens= 500)
            output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [25]:
def random_numbers(N = 100, minimum = 0, maximum = 999):
    return np.random.randint(minimum, maximum, size=(N, 1))

In [26]:
X_temps = random_numbers (N = 50, minimum=-20, maximum=200)

In [27]:
# Initialise our model
symbolic_model_C_to_F = SymbolicModel(sr_params={'constraints': {'sin':1, 'exp':1}, 'niterations' : 1000})

In [28]:
symbolic_model_C_to_F.fit(llm_C_to_F, X_temps)

No boxed number found. Trying again with more tokens.
Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.370e+06
Progress: 11855 / 31000 total iterations (38.242%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.811e+04  0.000e+00  y = x₀
3           4.590e+03  6.854e-01  y = x₀ * 1.9589
5           4.344e+03  2.732e-02  y = (x₀ + 15.5) * 1.7728
9           4.330e+03  6.803e-04  y = ((x₀ * 0.0010398) + 1.5588) * (x₀ + 20.15)
10          4.124e+03  4.819e-02  y = (x₀ * (sin(x₀) + 7.4295)) * 0.26173
11          2.794e+03  3.891e-01  y = inv(sin(x₀) + 0.1888) + (x₀ * 2.0244)
13          2.481e+03  5.916e-02  y = ((x₀ * 1.7999) + 33.902) + inv(sin(x₀) + 0.18878)
15          2.480e+03  1.371e-04  y = (inv(sin(x₀) + 0.1888) + ((x₀ + x₀) * 0.93322)) + 27.4...
                                      11
17          2.475e+03  9.538e-04  y = (x₀ * -0.15978) 

[ Info: Final population:
[ Info: Results saved to:


  - SR_output/SymbolicModel/dim0_1764000373/hall_of_fame.csv


It has learnt a decent approximation.

In [30]:
symbolic_model_C_to_F.get_equation(complexity=5)

'(x0 + 15.500419) * 1.7727895'